## Introduction : Transformers

### Modeles

Hugging Face : GPT2

### Evaluation

- BLEU/ROUGE Score
- BERT Score

### Bechmark

| Modèle                                | BLEU Score | ROUGE-L Score | BERTScore |
|---------------------------------------|------------|----------------|-----------|
| **GPT-2**          | 0.0115     | 0.0952         | 0.8339    |
| **GPT-2 AventIQ**                     | 0.0047     | 0.1063         | 0.8294    |
| **DistilGPT2 yroshan**                     | 0.0033     | 0.0972         |  0.8278    |


## Common code

In [1]:

import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import re


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-07 17:26:59.492893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746631619.664382   10341 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746631619.713360   10341 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746631620.097944   10341 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more tha

In [2]:
# Load dataset and preprocess
df = pd.read_csv("MovieDataThread.csv")
df = df.dropna(subset=["Script"])
df = df.sample(n=1000, random_state=42)


In [3]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
train_texts, test_texts = train_test_split(df["Script"].tolist(), test_size=0.2, random_state=42)
dataset = DatasetDict({
    "train": Dataset.from_dict({"Script": train_texts}),
    "test": Dataset.from_dict({"Script": test_texts})
})


## Hugging face Transformer (GPT2)

https://huggingface.co/openai-community/gpt2

### Model

In [8]:
# Load the tokenizer and model from Hugging Face (GPT2)
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [9]:
# Tokenize the dataset
def tokenize_function(example):
    encoding = tokenizer(example["Script"], truncation=True, padding="longest", max_length=256)
    encoding["labels"] = encoding["input_ids"].copy()
    return encoding

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["Script"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map: 100%|██████████| 200/200 [00:08<00:00, 22.32 examples/s]


### Train

In [ ]:
# Train the model (Arguments)
training_args = TrainingArguments(
    output_dir="./gpt2-script",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=5,
    logging_dir="./logs",
    save_strategy="no",
    fp16=False,
    lr_scheduler_type="linear",
    warmup_steps=100,
)


In [11]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [12]:
# Train the model
trainer.train()

# Save the model
trainer.save_model("./gpt2-script")

Epoch,Training Loss,Validation Loss
1,3.369500,3.111775
2,2.916900,3.124334
3,2.726900,3.168705
4,2.503200,3.215856
5,2.432400,3.255053


### Load model

In [16]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import evaluate

# Load the model for evaluation
model = GPT2LMHeadModel.from_pretrained("./gpt2-script")
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-script")

# Generate text
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### Evaluation

In [18]:
bleu_scores = []
rouge_scores = []
bert_scores = []

PROMPT_LENGTH = 200
GENERATION_MAX_LENGTH = 100

total = len(test_texts)
i = 1

print("Evaluating the model on the test set...")
for full_text in test_texts:
    print(f"Evaluating {i}/{total}...")
    prompt = full_text[:PROMPT_LENGTH]
    reference = full_text[PROMPT_LENGTH:PROMPT_LENGTH + GENERATION_MAX_LENGTH]

    generated = generate_text(prompt, max_new_tokens=GENERATION_MAX_LENGTH)
    generated = generated[PROMPT_LENGTH:PROMPT_LENGTH + GENERATION_MAX_LENGTH]

    # BLEU
    bleu_result = bleu.compute(predictions=[generated], references=[[reference]])
    bleu_scores.append(bleu_result["bleu"])

    # ROUGE
    rouge_result = rouge.compute(predictions=[generated], references=[reference])
    rouge_scores.append(rouge_result["rougeL"])

    # BERTScore
    bert_result = bertscore.compute(predictions=[generated], references=[reference], lang="en")
    bert_scores.append(bert_result["f1"][0])

    i += 1

print(f"Average BLEU score : {sum(bleu_scores) / len(bleu_scores):.6f}")
print(f"Average ROUGE score : {sum(rouge_scores) / len(rouge_scores):.6f}")
print(f"Average BERT score : {sum(bert_scores) / len(bert_scores):.6f}")


Evaluating the model on the test set...
Evaluating 1/200...
Evaluating 2/200...
Evaluating 3/200...
Evaluating 4/200...
Evaluating 5/200...
Evaluating 6/200...
Evaluating 7/200...
Evaluating 8/200...
Evaluating 9/200...
Evaluating 10/200...
Evaluating 11/200...
Evaluating 12/200...
Evaluating 13/200...
Evaluating 14/200...
Evaluating 15/200...
Evaluating 16/200...
Evaluating 17/200...
Evaluating 18/200...
Evaluating 19/200...
Evaluating 20/200...
Evaluating 21/200...
Evaluating 22/200...
Evaluating 23/200...
Evaluating 24/200...
Evaluating 25/200...
Evaluating 26/200...
Evaluating 27/200...
Evaluating 28/200...
Evaluating 29/200...
Evaluating 30/200...
Evaluating 31/200...
Evaluating 32/200...
Evaluating 33/200...
Evaluating 34/200...
Evaluating 35/200...
Evaluating 36/200...
Evaluating 37/200...
Evaluating 38/200...
Evaluating 39/200...
Evaluating 40/200...
Evaluating 41/200...
Evaluating 42/200...
Evaluating 43/200...
Evaluating 44/200...
Evaluating 45/200...
Evaluating 46/200...
Eva

# TODO

- Augmenter l'epoch
- max_length=128 ou 256

## Hugging face Transformer (AventIQ)

https://huggingface.co/AventIQ-AI/gpt-2-movie-script-writter

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM


# Load the tokenizer and model from Hugging Face (GPT2)
model_name = "AventIQ-AI/gpt-2-movie-script-writter"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
# Tokenize the dataset
def tokenize_function(example):
    encoding = tokenizer(example["Script"], truncation=True, padding="longest", max_length=256)
    encoding["labels"] = encoding["input_ids"].copy()
    return encoding

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["Script"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map: 100%|██████████| 200/200 [00:08<00:00, 22.23 examples/s]


### Train

In [6]:
# Train the model (Arguments)
training_args = TrainingArguments(
    output_dir="./aventiq-script",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=5,
    logging_dir="./logs",
    save_strategy="no",
    fp16=True,
    lr_scheduler_type="linear",
    warmup_steps=100,
)


In [7]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [8]:
# Train the model
trainer.train()

# Save the model
trainer.save_model("./aventiq-script")

Epoch,Training Loss,Validation Loss
1,3.432300,3.110733
2,2.916400,3.115852
3,2.729600,3.156087
4,2.501000,3.203796
5,2.428100,3.243665


In [9]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import evaluate

# Load the model for evaluation
model = GPT2LMHeadModel.from_pretrained("./aventiq-script")
tokenizer = GPT2Tokenizer.from_pretrained("./aventiq-script")

# Generate text
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### Evaluation

In [10]:
bleu_scores = []
rouge_scores = []
bert_scores = []

PROMPT_LENGTH = 200
GENERATION_MAX_LENGTH = 100

total = len(test_texts)
i = 1

print("Evaluating the model on the test set...")
for full_text in test_texts:
    print(f"Evaluating {i}/{total}...")
    prompt = full_text[:PROMPT_LENGTH]
    reference = full_text[PROMPT_LENGTH:PROMPT_LENGTH + GENERATION_MAX_LENGTH]

    generated = generate_text(prompt, max_new_tokens=GENERATION_MAX_LENGTH)
    generated = generated[PROMPT_LENGTH:PROMPT_LENGTH + GENERATION_MAX_LENGTH]

    # BLEU
    bleu_result = bleu.compute(predictions=[generated], references=[[reference]])
    bleu_scores.append(bleu_result["bleu"])

    # ROUGE
    rouge_result = rouge.compute(predictions=[generated], references=[reference])
    rouge_scores.append(rouge_result["rougeL"])

    # BERTScore
    bert_result = bertscore.compute(predictions=[generated], references=[reference], lang="en")
    bert_scores.append(bert_result["f1"][0])

    i += 1

print(f"Average BLEU score : {sum(bleu_scores) / len(bleu_scores):.6f}")
print(f"Average ROUGE score : {sum(rouge_scores) / len(rouge_scores):.6f}")
print(f"Average BERT score : {sum(bert_scores) / len(bert_scores):.6f}")


Evaluating the model on the test set...
Evaluating 1/200...


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating 2/200...
Evaluating 3/200...
Evaluating 4/200...
Evaluating 5/200...
Evaluating 6/200...
Evaluating 7/200...
Evaluating 8/200...
Evaluating 9/200...
Evaluating 10/200...
Evaluating 11/200...
Evaluating 12/200...
Evaluating 13/200...
Evaluating 14/200...
Evaluating 15/200...
Evaluating 16/200...
Evaluating 17/200...
Evaluating 18/200...
Evaluating 19/200...
Evaluating 20/200...
Evaluating 21/200...
Evaluating 22/200...
Evaluating 23/200...
Evaluating 24/200...
Evaluating 25/200...
Evaluating 26/200...
Evaluating 27/200...
Evaluating 28/200...
Evaluating 29/200...
Evaluating 30/200...
Evaluating 31/200...
Evaluating 32/200...
Evaluating 33/200...
Evaluating 34/200...
Evaluating 35/200...
Evaluating 36/200...
Evaluating 37/200...
Evaluating 38/200...
Evaluating 39/200...
Evaluating 40/200...
Evaluating 41/200...
Evaluating 42/200...
Evaluating 43/200...
Evaluating 44/200...
Evaluating 45/200...
Evaluating 46/200...
Evaluating 47/200...
Evaluating 48/200...
Evaluating 49/200...


## Hugging face Transformer (DistilGPT2)

https://huggingface.co/yroshan/moviescript

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "yroshan/moviescript"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [5]:
# Tokenize the dataset
def tokenize_function(example):
    encoding = tokenizer(example["Script"], truncation=True, padding="longest", max_length=256)
    encoding["labels"] = encoding["input_ids"].copy()
    return encoding

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["Script"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map: 100%|██████████| 200/200 [00:01<00:00, 144.35 examples/s]


In [14]:
# Train the model (Arguments)
training_args = TrainingArguments(
    output_dir="./distil-gpt2-script",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=5,
    logging_dir="./logs",
    save_strategy="no",
    fp16=True,
    lr_scheduler_type="linear",
    warmup_steps=100,
)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [16]:
# Train the model
trainer.train()

# Save the model
trainer.save_model("./distil-gpt2-script")

Epoch,Training Loss,Validation Loss
1,3.579100,3.322780
2,3.187900,3.322795
3,3.041700,3.351441
4,2.865500,3.377643
5,2.819900,3.405763


In [6]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import evaluate

# Load the model for evaluation
model = GPT2LMHeadModel.from_pretrained("./distil-gpt2-script")
tokenizer = GPT2Tokenizer.from_pretrained("./distil-gpt2-script")

# Generate text
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [7]:
bleu_scores = []
rouge_scores = []
bert_scores = []

PROMPT_LENGTH = 200
GENERATION_MAX_LENGTH = 100

total = len(test_texts)
i = 1

print("Evaluating the model on the test set...")
for full_text in test_texts:
    print(f"Evaluating {i}/{total}...")
    prompt = full_text[:PROMPT_LENGTH]
    reference = full_text[PROMPT_LENGTH:PROMPT_LENGTH + GENERATION_MAX_LENGTH]

    generated = generate_text(prompt, max_new_tokens=GENERATION_MAX_LENGTH)
    generated = generated[PROMPT_LENGTH:PROMPT_LENGTH + GENERATION_MAX_LENGTH]

    # BLEU
    bleu_result = bleu.compute(predictions=[generated], references=[[reference]])
    bleu_scores.append(bleu_result["bleu"])

    # ROUGE
    rouge_result = rouge.compute(predictions=[generated], references=[reference])
    rouge_scores.append(rouge_result["rougeL"])

    # BERTScore
    bert_result = bertscore.compute(predictions=[generated], references=[reference], lang="en")
    bert_scores.append(bert_result["f1"][0])

    i += 1

print(f"Average BLEU score : {sum(bleu_scores) / len(bleu_scores):.6f}")
print(f"Average ROUGE score : {sum(rouge_scores) / len(rouge_scores):.6f}")
print(f"Average BERT score : {sum(bert_scores) / len(bert_scores):.6f}")


Evaluating the model on the test set...
Evaluating 1/200...


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating 2/200...
Evaluating 3/200...
Evaluating 4/200...
Evaluating 5/200...
Evaluating 6/200...
Evaluating 7/200...
Evaluating 8/200...
Evaluating 9/200...
Evaluating 10/200...
Evaluating 11/200...
Evaluating 12/200...
Evaluating 13/200...
Evaluating 14/200...
Evaluating 15/200...
Evaluating 16/200...
Evaluating 17/200...
Evaluating 18/200...
Evaluating 19/200...
Evaluating 20/200...
Evaluating 21/200...
Evaluating 22/200...
Evaluating 23/200...
Evaluating 24/200...
Evaluating 25/200...
Evaluating 26/200...
Evaluating 27/200...
Evaluating 28/200...
Evaluating 29/200...
Evaluating 30/200...
Evaluating 31/200...
Evaluating 32/200...
Evaluating 33/200...
Evaluating 34/200...
Evaluating 35/200...
Evaluating 36/200...
Evaluating 37/200...
Evaluating 38/200...
Evaluating 39/200...
Evaluating 40/200...
Evaluating 41/200...
Evaluating 42/200...
Evaluating 43/200...
Evaluating 44/200...
Evaluating 45/200...
Evaluating 46/200...
Evaluating 47/200...
Evaluating 48/200...
Evaluating 49/200...
